# Try a contract in Great Expectations

Runs a dq-agent contract with Great Expectations 0.18.2, the version data-flow uses, so you can see whether it works before it goes to data-flow. Use it on an approved contract file, or paste the YAML the agent shows at the approval step to try a draft before approving it.

Two targets:

- **`"postgres"`**: the real table named in the contract, through the same `run_contract` the data-flow task calls. This is the test that counts. The connection comes from `DATABASE_DSN__datasets_1`, as for the agent's `profile_table`. Needs `uv sync --extra agents --extra postgres`.
- **a CSV or Parquet path**: a quick smoke test with GX's pandas engine. Regex is then Python's rather than Postgres's, and `expect_column_values_to_be_in_type_list` sees pandas dtypes rather than column types, so passing here does not guarantee passing in data-flow.

Nothing is stored. The GX context here is in memory, unlike data-flow's `gx_get_context()`, which writes results to `dataflow.ge_validation_store`.

In [ ]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# An approved contract file, relative to the repo root...
CONTRACT_FILE = "dataflow_kit/dq_contracts/synthetic.orders.yaml"
# ...or the YAML the agent shows at the approval step. Used instead of the file when not empty.
CONTRACT_YAML = """
"""

# "postgres" runs against the contract's schema.table; a file path runs on that file instead
TARGET = "data/synthetic/orders.csv"

## 1. Check the contract

The same checks data-flow's loader applies. A draft has no `approval` block yet, so a placeholder is added to let it run here; data-flow would reject it.

In [ ]:
import datetime

import yaml

from dataflow_kit import dq_validation as dq

text = CONTRACT_YAML if CONTRACT_YAML.strip() else (ROOT / CONTRACT_FILE).read_text(encoding="utf-8")
contract = yaml.safe_load(text)
if "approval" not in contract:
    print("Draft contract: adding a placeholder approval for this trial only")
    contract["approval"] = {
        "approved_by": "notebook-trial",
        "approved_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    }

problems = dq.contract_errors(contract)
if problems:
    raise ValueError("data-flow would reject this contract:\n- " + "\n- ".join(problems))
print(f"{contract['suite']['name']}: {len(contract['suite']['expectations'])} expectations, valid")

## 2. Run it

In [ ]:
import great_expectations as gx

context = gx.get_context()  # in memory: nothing is persisted

if TARGET == "postgres":
    from dq_agent.connectors import resolve_dsn

    result = dq.run_contract(
        context,
        contract,
        connection_string=resolve_dsn(),
        checkpoint_name="notebook-trial",
        run_name="notebook-trial",
        raise_on_failure=False,
    )
else:
    path = ROOT / TARGET
    source = context.sources.add_pandas(name="local")
    if path.suffix == ".parquet":
        asset = source.add_parquet_asset(name=path.stem, path=path)
    else:
        asset = source.add_csv_asset(name=path.stem, filepath_or_buffer=path)
    suite = context.add_expectation_suite(
        expectation_suite_name=contract["suite"]["name"],
        expectations=dq.expectation_configurations(contract),
    )
    checkpoint = context.add_checkpoint(
        name="notebook-trial",
        validator=context.get_validator(
            batch_request=asset.build_batch_request(),
            expectation_suite_name=suite.name,
        ),
    )
    result = checkpoint.run()

print("PASSED" if result["success"] else "FAILED")
print(dq.summarize(result))

## 3. Why each failure failed

Observed values, a sample of unexpected values, or the error GX hit. An error usually means the expectation does not fit the column (a wrong type name, a regex Postgres rejects) rather than bad data. This shows real cell values, so keep it out of anything you share.

In [ ]:
for validation in result.list_validation_results():
    for r in validation.results:
        if r.success:
            continue
        config = r.expectation_config
        print(f"{config.expectation_type} on {config.kwargs.get('column', '<table>')}")
        if r.exception_info and r.exception_info.get("raised_exception"):
            print("  error:", r.exception_info["exception_message"])
        details = r.result or {}
        if "observed_value" in details:
            print("  observed:", details["observed_value"])
        if details.get("partial_unexpected_list"):
            print("  unexpected (sample):", details["partial_unexpected_list"])